In [ ]:
from pyspark.sql.functions import col

# ENFORCE SCHEMA (Default)
# Create initial table
path_v1 = "/Volumes/example_data/delta_example/landing_zone/delta_demo/employees_v1.csv"
df_v1 = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(path_v1)

# Save as delta table in unity catalog
df_v1.write.format("delta").mode("overwrite").saveAsTable("example_data.delta_example.delta_employees")

# Schema Evolution
path_v2 = "/Volumes/example_data/delta_example/landing_zone/delta_demo/employees_2.csv"
df_v2 = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(path_v2)

try:
    # This will fail by default, Schema Enforcement
    df_v2.write.format("delta").mode("append").saveAsTable("example_data.delta_example.delta_employees")
except Exception as e:
    print("Write failed as expected: Schema Mismatch")

# Correct way: Explicitly allow evolution
df_v2.write.format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable("example_data.delta_example.delta_employees")

# Time Travel & Versioning
# History of the table
display(spark.sql("DESCRIBE HISTORY example_data.delta_example.delta_employees"))

# Querying version 0 (before the new employees and column were added)
print("Table as it looked in Version 0")
df_v0 = spark.read.format("delta").option("versionAsOf", 0).table("example_data.delta_example.delta_employees")
df_v0.show()

In [ ]:
%sql
-- OPTIMIZE - Takes thousands of tiny files and squashes them into a few large efficient files (Compaction)
-- Z-ORDER - Reorganizes the data so that related information is stored together physically, if you join on customer_id often, Z-ORDERing by that ID make your joins 10 X faster
-- VACUUM - Deletes old files that are no longer needed for current version
-- If you vacuum, you lose the ability to time travel back to those deleted version

OPTIMIZE example_data.delta_example.delta_employees ZORDER BY (emp_id);

-- clean up files older than 7 days
-- In free edition you may need to set a spark config to vacuum 0 hours for testing
-- SET delta.retentionDurationCheck.enabled = false;

VACUUM example_data.delta_example.delta_employees RETAIN 0 HOURS;